# NB02: Student Knowledge Distillation

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab - skip Drive mount.')


## RUN MODE — edit the cell below (only switch in this notebook)

| `XFAD_DRY_RUN` | Meaning |
|---|---|
| `'1'` | **Smoke test** — tiny samples, 1 epoch, fast sanity check (~minutes) |
| `'0'` | **Full paper run** — all data, full training, both student arms (~hours) |

On a **full run**, NB02–NB05 automatically process **baseline** and **saliency** students (everything NB06 needs).

Then: **Runtime → Run all** (restart session after changing the switch).


In [ ]:
import os

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  ONLY SWITCH IN THIS NOTEBOOK                                        ║
# ║  '0' = FULL PAPER RUN (all data, both student arms)                  ║
# ║  '1' = SMOKE TEST (tiny data, baseline student only)                 ║
# ╚══════════════════════════════════════════════════════════════════════╝
os.environ['XFAD_DRY_RUN'] = '0'


In [ ]:
import os
import sys
from pathlib import Path



def bootstrap_xfad_root() -> Path:
    """Locate repo root and put it on sys.path (works in Colab + local)."""
    candidates = []
    env = os.environ.get('XFAD_ROOT')
    if env:
        candidates.append(Path(env))
    candidates.extend([
        Path('/content/drive/MyDrive/XFAD'),
        Path.cwd(),
        Path.cwd().parent,
    ])
    candidates.extend(Path.cwd().resolve().parents)
    seen = set()
    for p in candidates:
        try:
            p = p.resolve()
        except OSError:
            continue
        if p in seen:
            continue
        seen.add(p)
        if (p / 'src' / 'config.py').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            os.environ.setdefault('XFAD_ROOT', str(p))
            return p
    raise FileNotFoundError(
        'XFAD repo root not found (expected src/config.py). '
        'In Colab: mount Drive first, then either keep the project at '
        '/content/drive/MyDrive/XFAD or set '
        "os.environ['XFAD_ROOT'] = '/content/drive/MyDrive/<your>/XFAD'."
    )

REPO_ROOT = bootstrap_xfad_root()
print('REPO_ROOT:', REPO_ROOT)

import src.config as config
if not hasattr(config, 'KD_WEIGHT_SALIENCY'):
    config.KD_WEIGHT_SALIENCY = 0.3
    print('Patched KD_WEIGHT_SALIENCY=0.3 in memory (stale Drive config.py).')
KD_WEIGHT_SALIENCY = config.KD_WEIGHT_SALIENCY

from src.utils.io import set_seed
set_seed(42)

from src.utils.io import resolve_dry_run, print_run_banner, ensure_dir
DRY_RUN = resolve_dry_run()
import json
import torch
import torch.nn as nn
from src.data.loader_utils import make_loader
from src.config import MODEL_DIR, RESULTS_DIR, KD_TEMPERATURE_DEFAULT, KD_WEIGHT_HARD, KD_WEIGHT_RESPONSE, KD_WEIGHT_FEATURE, MULTITASK_FALLBACK_AUROC_MARGIN, STUDENT_TRUNK_CHANNELS, STUDENT_EPOCHS, EARLY_STOPPING_PATIENCE, USE_CLASS_WEIGHTS
from src.data.mitbih import MITBIHDataset
from src.data.ptbxl import PTBXLDataset
from src.models.student import MultiTaskStudent
from src.models.teacher_a import TeacherA
from src.models.teacher_b import TeacherB
from src.models.kd_loss import kd_loss
from src.models.saliency_kd import saliency_kd_term

device = torch.device('cuda' if torch.cuda.is_available() and not DRY_RUN else 'cpu')

from src.utils.io import student_variants_for_run, print_run_banner
STUDENT_VARIANTS_TO_RUN = student_variants_for_run(DRY_RUN)
print_run_banner(notebook='NB02_Student_KD', dry_run=DRY_RUN, variants=STUDENT_VARIANTS_TO_RUN)


In [ ]:
from src.config import MITBIH_DS1_TRAIN, MITBIH_DS1_VAL, PTBXL_TRAIN_FOLDS, PTBXL_VAL_FOLD
from src.utils.class_weights import class_weights_from_labels

mit_ds = MITBIHDataset(MITBIH_DS1_TRAIN, dry_run=DRY_RUN, use_cache=True)
ptb_ds = PTBXLDataset(PTBXL_TRAIN_FOLDS, dry_run=DRY_RUN, use_cache=True)
batch_size = 2 if DRY_RUN else 64
mit_loader = make_loader(mit_ds, batch_size=batch_size, shuffle=True, dry_run=DRY_RUN)
ptb_loader = make_loader(ptb_ds, batch_size=batch_size, shuffle=True, dry_run=DRY_RUN)
mit_val_loader = make_loader(MITBIHDataset(MITBIH_DS1_VAL, dry_run=DRY_RUN, use_cache=True), batch_size=batch_size, shuffle=False, dry_run=DRY_RUN)
ptb_val_loader = make_loader(PTBXLDataset([PTBXL_VAL_FOLD], dry_run=DRY_RUN, use_cache=True), batch_size=batch_size, shuffle=False, dry_run=DRY_RUN)

teacher_a = TeacherA().to(device)
teacher_b = TeacherB().to(device)
teacher_a.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'teacher_a.pt'), map_location=device))
teacher_b.load_state_dict(torch.load(os.path.join(MODEL_DIR, 'teacher_b.pt'), map_location=device))
teacher_a.eval(); teacher_b.eval()

if USE_CLASS_WEIGHTS and not DRY_RUN:
    w_a = class_weights_from_labels([y for _, y in mit_ds.samples], num_classes=5).to(device)
    w_b = class_weights_from_labels([y for _, y in ptb_ds.samples], num_classes=12).to(device)
    print('Using inverse-frequency class weights in KD hard-loss term')
else:
    w_a = w_b = None

_probe = MultiTaskStudent(c_internal=STUDENT_TRUNK_CHANNELS).to(device)
_aa = nn.Conv1d(STUDENT_TRUNK_CHANNELS, 128, 1).to(device)
xa, ya = next(iter(mit_loader))
xb, yb = next(iter(ptb_loader))
xa, ya, xb, yb = xa.to(device), ya.to(device), xb.to(device), yb.to(device)
with torch.no_grad():
    ta_logits, ta_feat = teacher_a(xa, return_features=True)
tb_logits, tb_feat = teacher_b(xb, return_features=True)
sa_logits, sa_feat = _probe(xa, 'A', return_features=True)
loss = kd_loss(sa_logits, sa_feat, ta_logits, ta_feat, ya, _aa, T=KD_TEMPERATURE_DEFAULT,
               w_hard=KD_WEIGHT_HARD, w_resp=KD_WEIGHT_RESPONSE, w_feat=KD_WEIGHT_FEATURE, class_weight=w_a)
assert torch.isfinite(loss) and loss.item() > 0
print('KD unit test passed:', float(loss.detach()))


In [ ]:
for VARIANT in STUDENT_VARIANTS_TO_RUN:
    print('\n' + '=' * 60 + f'\nStudent variant: {VARIANT}\n' + '=' * 60)
    from src.utils.io import student_checkpoint_path, variant_result_path
    from src.utils.early_stopping import EarlyStopping
    STUDENT_PATH = student_checkpoint_path(VARIANT)
    print(f'Checkpoint: {STUDENT_PATH}')
    if VARIANT == 'saliency':
        print(f'Saliency-matching KD active (weight={KD_WEIGHT_SALIENCY}). Epochs take ~2-3x longer.')

    student = MultiTaskStudent(c_internal=STUDENT_TRUNK_CHANNELS).to(device)
    adapter_a = nn.Conv1d(STUDENT_TRUNK_CHANNELS, 128, 1).to(device)
    adapter_b = nn.Conv1d(STUDENT_TRUNK_CHANNELS, 256, 1).to(device)
    optimizer = torch.optim.Adam(
        list(student.parameters()) + list(adapter_a.parameters()) + list(adapter_b.parameters()),
        lr=1e-3,
    )
    early_stop = EarlyStopping(patience=EARLY_STOPPING_PATIENCE)
    history = []

    def eval_kd_val():
        student.eval(); adapter_a.eval(); adapter_b.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for batch_a, batch_b in zip(mit_val_loader, ptb_val_loader):
                xa, ya = batch_a; xb, yb = batch_b
                xa, ya, xb, yb = xa.to(device), ya.to(device), xb.to(device), yb.to(device)
                sa_logits, sa_feat = student(xa, 'A', return_features=True)
                sb_logits, sb_feat = student(xb, 'B', return_features=True)
                ta_logits, ta_feat = teacher_a(xa, return_features=True)
                tb_logits, tb_feat = teacher_b(xb, return_features=True)
                loss = kd_loss(sa_logits, sa_feat, ta_logits, ta_feat, ya, adapter_a, T=KD_TEMPERATURE_DEFAULT,
                               w_hard=KD_WEIGHT_HARD, w_resp=KD_WEIGHT_RESPONSE, w_feat=KD_WEIGHT_FEATURE, class_weight=w_a)
                loss = loss + kd_loss(sb_logits, sb_feat, tb_logits, tb_feat, yb, adapter_b, T=KD_TEMPERATURE_DEFAULT,
                                      w_hard=KD_WEIGHT_HARD, w_resp=KD_WEIGHT_RESPONSE, w_feat=KD_WEIGHT_FEATURE, class_weight=w_b)
                if VARIANT == 'saliency':
                    l_sal, _, _ = saliency_kd_term(student, teacher_a, xa, ya,
                        student_layer=MultiTaskStudent.LAST_CONV, teacher_layer=TeacherA.LAST_CONV)
                    loss = loss + KD_WEIGHT_SALIENCY * l_sal
                total += float(loss); n += 1
                if DRY_RUN: break
        student.train(); adapter_a.train(); adapter_b.train()
        return total / max(n, 1)

    epochs = 1 if DRY_RUN else STUDENT_EPOCHS
    for ep in range(epochs):
        saliency_running, n_batches = 0.0, 0
        train_running, train_n = 0.0, 0
        for i, (batch_a, batch_b) in enumerate(zip(mit_loader, ptb_loader)):
            xa, ya = batch_a; xb, yb = batch_b
            xa, ya, xb, yb = xa.to(device), ya.to(device), xb.to(device), yb.to(device)
            optimizer.zero_grad()
            sa_logits, sa_feat = student(xa, 'A', return_features=True)
            sb_logits, sb_feat = student(xb, 'B', return_features=True)
            with torch.no_grad():
                ta_logits, ta_feat = teacher_a(xa, return_features=True)
                tb_logits, tb_feat = teacher_b(xb, return_features=True)
            la = kd_loss(sa_logits, sa_feat, ta_logits, ta_feat, ya, adapter_a, T=KD_TEMPERATURE_DEFAULT,
                         w_hard=KD_WEIGHT_HARD, w_resp=KD_WEIGHT_RESPONSE, w_feat=KD_WEIGHT_FEATURE, class_weight=w_a)
            lb = kd_loss(sb_logits, sb_feat, tb_logits, tb_feat, yb, adapter_b, T=KD_TEMPERATURE_DEFAULT,
                         w_hard=KD_WEIGHT_HARD, w_resp=KD_WEIGHT_RESPONSE, w_feat=KD_WEIGHT_FEATURE, class_weight=w_b)
            loss = la + lb
            if VARIANT == 'saliency':
                l_sal, _, _ = saliency_kd_term(
                    student, teacher_a, xa, ya,
                    student_layer=MultiTaskStudent.LAST_CONV,
                    teacher_layer=TeacherA.LAST_CONV,
                )
                loss = loss + KD_WEIGHT_SALIENCY * l_sal
                saliency_running += float(l_sal); n_batches += 1
            train_running += float(loss.detach()); train_n += 1
            loss.backward(); optimizer.step()
            if DRY_RUN: break
        val_loss = eval_kd_val()
        row = {
            'epoch': ep + 1,
            'train_kd': train_running / max(train_n, 1),
            'val_kd': val_loss,
            'mean_saliency_loss': (saliency_running / n_batches) if n_batches else None,
        }
        history.append(row)
        msg = f'epoch {ep+1} done | train_kd={row["train_kd"]:.4f} | val_kd={val_loss:.4f}'
        if n_batches:
            msg += f' | mean saliency loss {saliency_running / n_batches:.4f}'
        print(msg)
        if early_stop.step(val_loss, student, adapter_a, adapter_b):
            print(f'Early stopping at epoch {ep+1} (patience={EARLY_STOPPING_PATIENCE})')
            break

    early_stop.restore(student, adapter_a, adapter_b)
    ensure_dir(MODEL_DIR)
    ensure_dir(RESULTS_DIR)
    torch.save(student.state_dict(), STUDENT_PATH)
    best_row = min(history, key=lambda r: r['val_kd'])
    history_path = variant_result_path('student_training_history', VARIANT)
    with open(history_path, 'w', encoding='utf-8') as f:
        json.dump({
            'dry_run': DRY_RUN,
            'student_variant': VARIANT,
            'max_epochs': epochs,
            'early_stopping_patience': EARLY_STOPPING_PATIENCE,
            'best_val_kd': early_stop.best,
            'best_epoch': best_row['epoch'],
            'stopped_epoch': history[-1]['epoch'] if history else None,
            'epochs': history,
        }, f, indent=2)
    print(f'Student saved to {STUDENT_PATH}. Params:', sum(p.numel() for p in student.parameters()),
          f'| best val_kd={early_stop.best:.4f} (epoch {best_row["epoch"]})')
    print('Training history saved to', history_path)
print('Negative-transfer margin (pre-registered):', MULTITASK_FALLBACK_AUROC_MARGIN)
